In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


df = pd.read_csv('lightning_logs/version_0/metrics.csv')


plt.style.use('ggplot')
sns.set(font_scale=1.2)


fig, axs = plt.subplots(2, 1, figsize=(14, 12))


train_data = df[df['train_loss_epoch'].notna()]
val_data = df[df['val_loss'].notna()]

axs[0].plot(train_data['epoch'], train_data['train_loss_epoch'], 'b-', label='Training Loss')
axs[0].plot(val_data['epoch'], val_data['val_loss'], 'r-', label='Validation Loss')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Loss')
axs[0].set_title('Training and Validation Loss')
axs[0].set_ylim(0, 0.5)  
axs[0].legend()
axs[0].grid(True)


test_data = df[df['test_accuracy'].notna()]
axs[1].plot(test_data['epoch'], test_data['test_accuracy'], 'g-o', label='Test Accuracy')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Accuracy')
axs[1].set_title('Test Accuracy')
axs[1].grid(True)
axs[1].legend()


if not test_data.empty:
    final_accuracy = test_data['test_accuracy'].iloc[-1]
    max_accuracy = test_data['test_accuracy'].max()
    max_accuracy_epoch = test_data.loc[test_data['test_accuracy'].idxmax()]['epoch']

    
    text = f"Final Accuracy: {final_accuracy:.4f}\nMax Accuracy: {max_accuracy:.4f} (Epoch {int(max_accuracy_epoch)})"
    axs[1].annotate(text, xy=(0.02, 0.02), xycoords='axes fraction',
                    bbox=dict(boxstyle="round,pad=0.5", fc="yellow", alpha=0.5))


plt.tight_layout()
plt.savefig('./images/training_curves.png', dpi=300)
plt.show()


step_data = df[df['train_loss_step'].notna()]
plt.figure(figsize=(14, 6))
plt.plot(step_data['step'], step_data['train_loss_step'], 'b-', alpha=0.5)
plt.xlabel('Step')
plt.ylabel('Training Loss')
plt.title('Training Loss per Step')
plt.ylim(0, 0.5)  
plt.grid(True)
plt.savefig('./images/training_loss_steps.png', dpi=300)
plt.show()

if not test_data.empty:
    print(f"Final Test Accuracy: {final_accuracy:.4f}")
    print(f"Maximum Test Accuracy: {max_accuracy:.4f} (at Epoch {int(max_accuracy_epoch)})")

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
acc_path = 'ckpt-SLEEP_03_SHHS/FAST_0/D0-32-96-100-90-4=15-185/pre-epoacc.npy'
def load_epoch_acc(path):
    acc = np.load(path)
    return acc

acc = load_epoch_acc(acc_path)
mean_acc = acc.mean(axis=0)
def plot_acc(acc):
    plt.plot(acc)
    plt.show()
plot_acc(mean_acc)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

result_path = 'outputs/multi_source_ds-num=16/whisper_icl_20250802_083127/test_results.pkl'
with open(result_path, 'rb') as f:
    results = pickle.load(f)

stage1 = results.get('stage1_test_results', {})
stage2 = results.get('stage2_test_results', {})

normal_scenes = ['first_stage', 'second_stage', 'no_support']
cross_scenes  = ['cross_ds_first_stage', 'cross_ds_second_stage', 'cross_ds_no_support']
scene_tag_map = {
    'Stage-1': stage1,
    '8-shot'  : stage2.get('first_stage', {}),
    '5-shot' : stage2.get('second_stage', {}),
    'No-Sup' : stage2.get('no_support', {})
}
cross_scene_tag_map = {
    '8-shot'  : stage2.get('cross_ds_first_stage', {}),
    '5-shot' : stage2.get('cross_ds_second_stage', {}),
    'No-Sup' : stage2.get('cross_ds_no_support', {})
}
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  

def plot_grouped_bar(ax, datasets, data_dict_per_tag, title):
    tags = list(data_dict_per_tag.keys())
    bar_w  = 0.15
    x_base = np.arange(len(datasets))

    for i, tag in enumerate(tags):
        ys = [data_dict_per_tag[tag].get(ds, 0) for ds in datasets]
        
        bars = ax.bar(
            x_base + (i - (len(tags)-1)/2)*bar_w,
            ys,
            width=bar_w,
            color=colors[i],
            label=tag,
        )

        
        ax.bar_label(
            bars,
            labels=[f'{y:.3f}' if y else '' for y in ys],
            padding=3,          
            fontsize=7,
            rotation=90,        
            label_type='edge'   
        )

    ax.set_xticks(x_base)
    ax.set_xticklabels(datasets, rotation=45, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.legend(title='Scene', fontsize=9, loc='lower right')


normal_datasets = sorted(
    set(stage1.keys()) |
    set().union(*[stage2.get(s, {}).keys() for s in normal_scenes])
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_grouped_bar(
    axes[0],
    normal_datasets,
    scene_tag_map,
    'Normal Datasets'
)


cross_datasets = sorted(
    set().union(*[stage2.get(s, {}).keys() for s in cross_scenes])
)
plot_grouped_bar(
    axes[1],
    cross_datasets,
    cross_scene_tag_map,
    'Cross-Datasets'
)

plt.tight_layout()
plt.show()

# Finetune Comparison

In [ ]:
import pickle
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

pkl_path = Path("outputs/multi_source_finetune_MI_02_ShanghaiU/whisper_icl_20250707_235747/finetune_results.pkl")  

with open(pkl_path, 'rb') as f:
    results = pickle.load(f)

pre = results.get('pre_finetune', {})
post = results.get('post_finetune', {})

datasets = sorted(set(pre) | set(post))
pre_vals  = [pre.get(ds, 0.0)  for ds in datasets]
post_vals = [post.get(ds, 0.0) for ds in datasets]

x = np.arange(len(datasets))
width = 0.35

plt.figure(figsize=(max(6, len(datasets) * 0.8), 4))
plt.bar(x - width/2, pre_vals,  width, label='Pre-train',  color='#4C72B0')
plt.bar(x + width/2, post_vals, width, label='Post-finetune', color='#55A868')

plt.ylabel('Accuracy')
plt.title('Pre-train vs Post-finetune Accuracy by Dataset')
plt.xticks(x, datasets, rotation=45, ha='right')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()